In [ ]:
from langgraph.graph import StateGraph, START
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

search_tool = DuckDuckGoSearchRun()

@tool
def calculator(
    first_num: float,
    second_num: float,
    operation: str
) -> dict:
    """It performs basic mathematical operations."""

    try:
        if operation == "add":
            result = first_num + second_num

        elif operation == "sub":
            result = first_num - second_num

        elif operation == "multiply":
            result = first_num * second_num

        elif operation == "div":
            if second_num == 0:
                return {
                    "error": "Division by zero is not allowed"
                }

            result = first_num / second_num

        else:
            return {
                "error": f"Unsupported operation '{operation}'"
            }

        return {
            "first_num": first_num,
            "second_num": second_num,
            "operation": operation,
            "result": result
        }

    except Exception as e:
        return {
            "error": str(e)
        }


tools = [
    search_tool,
    calculator
]

llm_with_tools = llm.bind_tools(tools)

class ChatState(TypedDict):
    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]

def chat_node(state: ChatState):
    messages = state["messages"]

    response = llm_with_tools.invoke(messages)

    return {
        "messages": [response]
    }

tool_node = ToolNode(tools)

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges(
    "chat_node",
    tools_condition
)

graph.add_edge("tools", "chat_node")

work_flow = graph.compile()

def get_final_answer(question):
    result = work_flow.invoke({
        "messages": [
            HumanMessage(content=question)
        ]
    })

    return result["messages"][-1].content

print(get_final_answer("What is the price of Apple's iPhone 17 and how muchmoney we need to buy 50 iphones?"))


# -----------------------------
# 7. Helper
# -----------------------------
def retrieve_all_threads():
    all_threads = set()
    for checkpoint in checkpointer.list(None):
        all_threads.add(checkpoint.config["configurable"]["thread_id"])
    return list(all_threads)

**iPhone 17 price (U.S. retail)**  
- Base‑model (128 GB) starts at **$799**.

**Cost for 50 iPhones**  
- 50 × $799 = **$39,950**.

---

### Quick note on other markets

| Region | Approx. price (per unit) | 50‑unit cost |
|--------|--------------------------|--------------|
| United States | $799 | $39,950 |
| United Kingdom | £799 (≈ $1,050) | ≈ $52,500 |
| Pakistan | Rs 400,000 (≈ $2,500) | ≈ Rs 20,000,000 (≈ $125,000) |

> **Tip:** Prices can vary by storage size, carrier, and local taxes. If you’re buying in a specific country, it’s a good idea to check the latest local listings or the Apple website for that country.
